In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import requests

# 1. ADD YOUR API KEY
API_KEY = "309B8432-5CDB-300F-A319-E0501F346610"

url = "http://quickstats.nass.usda.gov/api/api_GET/"
params = {
    'key': API_KEY,
    'source_desc': 'SURVEY',
    'sector_desc': 'CROPS',
    'commodity_desc': 'WHEAT',
    'class_desc': 'WINTER',
    'statisticcat_desc': 'YIELD',
    'agg_level_desc': 'COUNTY',
    'year__GE': '2000',
    'format': 'JSON'
}

print("Downloading National Winter Wheat Yields from USDA...")
response = requests.get(url, params=params)
data = response.json()
df_yield = pd.DataFrame(data['data'])

# Clean and format
df_yield = df_yield[['year', 'state_alpha', 'county_name', 'state_fips_code', 'county_code', 'Value']]
df_yield.columns = ['Year', 'State', 'County', 'State_FIPS', 'County_FIPS', 'Yield_BU_per_Acre']

# Create master 5-digit FIPS code
df_yield['FIPS'] = df_yield['State_FIPS'] + df_yield['County_FIPS']

# Remove missing data and convert to numeric
df_yield = df_yield[~df_yield['Yield_BU_per_Acre'].astype(str).str.contains('[a-zA-Z]')]
df_yield['Yield_BU_per_Acre'] = pd.to_numeric(df_yield['Yield_BU_per_Acre'].astype(str).str.replace(',', ''))

print(f"National Yield Data Acquired: {len(df_yield)} rows successfully loaded.")

National Yield Data Acquired: 49644 rows successfully loaded.


In [ ]:
print("Filtering for the Top 300 most consistent counties...")

# Count how many years of data each county reported
county_consistency = df_yield.groupby(['FIPS', 'State', 'County']).size().reset_index(name='Years_of_Data')

# Sort to get the counties with the most complete records
county_consistency = county_consistency.sort_values(by='Years_of_Data', ascending=False)

# Slice the Top 300
top_300_roster = county_consistency.head(300).copy()

print("Top 300 Roster successfully created and saved to memory.")

Filtering for the Top 300 most consistent counties...
Top 300 Roster successfully created and saved to memory.


In [ ]:
import pandas as pd

print("Aligning Top 300 Counties with official US Census coordinates...")

# 1. THE BULLETPROOF CENSUS URL
# This is the official US Government Census dataset for county coordinates
census_url = "https://www2.census.gov/geo/docs/reference/cenpop2020/county/CenPop2020_Mean_CO.txt"

# 2. READ AND CLEAN THE CENSUS DATA
# The Census uses a comma-separated text file
df_coords = pd.read_csv(census_url, dtype={'STATEFP': str, 'COUNTYFP': str})

# The Census splits the FIPS code into State (STATEFP) and County (COUNTYFP)
# We combine them to create our master 5-digit FIPS code
df_coords['FIPS'] = df_coords['STATEFP'] + df_coords['COUNTYFP']

# Rename the columns so they match our exact pipeline format
df_coords = df_coords.rename(columns={'LATITUDE': 'Latitude', 'LONGITUDE': 'Longitude'})

# 3. MERGE WITH OUR TOP 300 ROSTER
# We join our Top 300 wheat counties with the official Census coordinates
target_roster = pd.merge(top_300_roster, df_coords[['FIPS', 'Latitude', 'Longitude']], on='FIPS', how='inner')

# Convert to a dictionary list for the NASA weather loop
targets = target_roster.to_dict('records')

print(f"✅ Coordinates successfully mapped for {len(targets)} unique counties!")
display(target_roster.head())

Aligning Top 300 Counties with official US Census coordinates...
✅ Coordinates successfully mapped for 264 unique counties!


,FIPS,State,County,Years_of_Data,Latitude,Longitude
0,30073,MT,PONDERA,114,48.211424,-112.157767
1,30099,MT,TETON,110,47.762524,-112.039341
2,30013,MT,CASCADE,108,47.491558,-111.306586
3,30003,MT,BIG HORN,103,45.568552,-107.499673
4,30111,MT,YELLOWSTONE,97,45.789945,-108.546051


In [ ]:
import time
import os
import requests
import pandas as pd
from google.colab import drive

# 1. MOUNT DRIVE & SETUP CHECKPOINTS
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/wheat_yield_usa_checkpoints/'
os.makedirs(save_dir, exist_ok=True)

all_weather = []
batch_size = 50

print(f"Starting NASA API Extraction for {len(targets)} counties...")

# 2. THE EXTRACTION LOOP
for i, target in enumerate(targets):
    batch_file = f"{save_dir}weather_batch_{(i//batch_size) + 1}.csv"

    # Skip batches that are already finished
    if os.path.exists(batch_file) and (i % batch_size == 0):
        print(f"Skipping batch {(i//batch_size) + 1} (Already exists on Drive)...")
        continue

    print(f"[{i+1}/{len(targets)}] Pulling weather for {target['County']}, {target['State']}...")

    url = (
        f"https://power.larc.nasa.gov/api/temporal/monthly/point?"
        f"parameters=T2M_MAX,PRECTOTCORR&community=AG&"
        f"longitude={target['Longitude']}&latitude={target['Latitude']}&"
        f"start=2000&end=2023&format=JSON"
    )

    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            data = response.json()
            tmax = data['properties']['parameter']['T2M_MAX']
            precip = data['properties']['parameter']['PRECTOTCORR']

            for year_month in tmax.keys():
                if not year_month.endswith('13'): # Skip annual averages
                    all_weather.append({
                        'FIPS': target['FIPS'],
                        'Year': int(year_month[:4]),
                        'Month': int(year_month[4:]),
                        'Max_Temp_C': tmax[year_month],
                        'Precip_mm': precip[year_month]
                    })
        else:
            print(f"API Error {response.status_code} for {target['County']}")

    except Exception as e:
        print(f"Connection failed for {target['County']}: {e}")

    # Prevent NASA server blocking
    time.sleep(1)

    # Save checkpoint to Drive
    if (i + 1) % batch_size == 0 or (i + 1) == len(targets):
        if all_weather:
            temp_df = pd.DataFrame(all_weather)
            temp_df.to_csv(batch_file, index=False)
            print(f"Checkpoint saved to Drive: {batch_file}")
            all_weather = []

print("All Weather Data Extracted and Saved securely.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting NASA API Extraction for 264 counties...
[1/264] Pulling weather for PONDERA, MT...
[2/264] Pulling weather for TETON, MT...
[3/264] Pulling weather for CASCADE, MT...
[4/264] Pulling weather for BIG HORN, MT...
[5/264] Pulling weather for YELLOWSTONE, MT...
[6/264] Pulling weather for GALLATIN, MT...
[7/264] Pulling weather for PARK, MT...
[8/264] Pulling weather for CHOUTEAU, MT...
[9/264] Pulling weather for FLATHEAD, MT...
[10/264] Pulling weather for LAKE, MT...
[11/264] Pulling weather for BLAINE, MT...
[12/264] Pulling weather for JUDITH BASIN, MT...
[13/264] Pulling weather for BROADWATER, MT...
[14/264] Pulling weather for MORTON, KS...
[15/264] Pulling weather for UMATILLA, OR...
[16/264] Pulling weather for POWER, ID...
[17/264] Pulling weather for CARIBOU, ID...
[18/264] Pulling weather for WASCO, OR...
[19/264] Pulling weather for FERGUS,

In [ ]:
import pandas as pd
import glob
import os

print("Step 1: Loading all monthly weather checkpoints...")
save_dir = '/content/drive/MyDrive/wheat_yield_usa_checkpoints/'
all_files = glob.glob(os.path.join(save_dir, "weather_batch_*.csv"))

df_weather_raw = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

print("Step 2: Applying Winter Wheat 'Harvest Year' Logic...")
# If the month is Sept (9) or later, that weather affects NEXT year's harvest.
df_weather_raw['Harvest_Year'] = df_weather_raw.apply(
    lambda row: int(row['Year']) + 1 if row['Month'] >= 9 else int(row['Year']),
    axis=1
)

print("Step 3: Engineering Biological Features...")
# Define our specific growth periods
def calculate_monthly_features(group):
    # Growing season: Sept (9) to June (6)
    gs = group[group['Month'].isin([9, 10, 11, 12, 1, 2, 3, 4, 5, 6])]
    # Sowing months: Sept, Oct
    sowing = group[group['Month'].isin([9, 10])]
    # Terminal heat months: May, June (Grain filling stage for US Winter Wheat)
    terminal = group[group['Month'].isin([5, 6])]

    return pd.Series({
        'Avg_Season_Tmax': gs['Max_Temp_C'].mean(),
        'Total_Season_Rain': gs['Precip_mm'].sum(),
        'Sowing_Rainfall': sowing['Precip_mm'].sum(),
        'Terminal_Heat_Tmax': terminal['Max_Temp_C'].mean()
    })

# Apply the calculations per County per Harvest Year
df_features = df_weather_raw.groupby(['FIPS', 'Harvest_Year']).apply(calculate_monthly_features).reset_index()

print("Step 4: Merging with USDA Yields & Converting Units...")
# Ensure FIPS is string for perfect merge
df_features['FIPS'] = df_features['FIPS'].astype(str).str.zfill(5)
df_yield['FIPS'] = df_yield['FIPS'].astype(str).str.zfill(5)

# Convert Yield from US Bushels/Acre to standard Kg/Hectare (1 BU/Acre = 67.25 Kg/Ha)
df_yield['Yield_Kg_Ha'] = df_yield['Yield_BU_per_Acre'] * 67.25

# Merge yields with our engineered weather features
df_usa = pd.merge(df_yield, df_features, left_on=['FIPS', 'Year'], right_on=['FIPS', 'Harvest_Year'], how='inner')

print("Step 5: Calculating Equalizer Features (Deviations)...")
# Calculate historical county averages to create deviations
county_means = df_usa.groupby('FIPS').agg(
    Historical_Mean_Rain=('Total_Season_Rain', 'mean'),
    Historical_Mean_Tmax=('Avg_Season_Tmax', 'mean')
).reset_index()

df_usa = pd.merge(df_usa, county_means, on='FIPS', how='left')

# Calculate Deviations (Crucial for comparing USA to India)
df_usa['Rainfall_Deviation'] = df_usa['Total_Season_Rain'] - df_usa['Historical_Mean_Rain']
df_usa['Temperature_Anomaly'] = df_usa['Avg_Season_Tmax'] - df_usa['Historical_Mean_Tmax']

# Calculate Yield Lag 1
df_usa = df_usa.sort_values(by=['FIPS', 'Year'])
df_usa['Yield_Lag1'] = df_usa.groupby('FIPS')['Yield_Kg_Ha'].shift(1)
df_usa = df_usa.dropna(subset=['Yield_Lag1']).reset_index(drop=True)

print("Step 6: Final Cleanup...")
# Rename and organize columns
df_usa = df_usa.rename(columns={'FIPS': 'Dist Code', 'County': 'District', 'Yield_Kg_Ha': 'Yield'})
df_usa['Country'] = 'USA'

final_columns = [
    'Country', 'Dist Code', 'State', 'District', 'Year', 'Yield', 'Yield_Lag1',
    'Avg_Season_Tmax', 'Total_Season_Rain', 'Sowing_Rainfall', 'Terminal_Heat_Tmax',
    'Temperature_Anomaly', 'Rainfall_Deviation'
]
df_final_usa = df_usa[final_columns]

# Save the final USA dataset
final_path = '/content/drive/MyDrive/final_usa_wheat_dataset.csv'
df_final_usa.to_csv(final_path, index=False)

print(f"✅ Final USA Dataset Engineered! Saved to {final_path}")
display(df_final_usa.head())

Step 1: Loading all monthly weather checkpoints...
Step 2: Applying Winter Wheat 'Harvest Year' Logic...
Step 3: Engineering Biological Features...


/tmp/ipykernel_427/1670784362.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_features = df_weather_raw.groupby(['FIPS', 'Harvest_Year']).apply(calculate_monthly_features).reset_index()


Step 4: Merging with USDA Yields & Converting Units...
Step 5: Calculating Equalizer Features (Deviations)...
Step 6: Final Cleanup...
✅ Final USA Dataset Engineered! Saved to /content/drive/MyDrive/final_usa_wheat_dataset.csv


,Country,Dist Code,State,District,Year,Yield,Yield_Lag1,Avg_Season_Tmax,Total_Season_Rain,Sowing_Rainfall,Terminal_Heat_Tmax,Temperature_Anomaly,Rainfall_Deviation
0,USA,06095,CA,SOLANO,2000,5991.975,5198.425,28.015,12.99,0.00,39.96,-0.302025,-1.54625
1,USA,06095,CA,SOLANO,2000,1956.975,5991.975,28.015,12.99,0.00,39.96,-0.302025,-1.54625
2,USA,06095,CA,SOLANO,2001,4875.625,1956.975,29.611,11.17,2.11,39.74,1.293975,-3.36625
3,USA,06095,CA,SOLANO,2001,5649.000,4875.625,29.611,11.17,2.11,39.74,1.293975,-3.36625
4,USA,06095,CA,SOLANO,2001,1008.750,5649.000,29.611,11.17,2.11,39.74,1.293975,-3.36625


In [1]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define file paths for the USA dataset
input_path = '/content/drive/MyDrive/wheat_yield_india/dataset/final_usa_wheat_dataset.csv'
output_path = '/content/drive/MyDrive/wheat_yield_india/dataset/final_usa_wheat_dataset_moderate.csv'

# 3. Load and sort the dataset chronologically per district
df = pd.read_csv(input_path)
df = df.sort_values(["Dist Code", "Year"]).reset_index(drop=True)

# 4. Temporal Engineering: Lag 2 and Rolling Statistics
df["Yield_Lag2"] = df.groupby("Dist Code")["Yield"].shift(2)
district_median_yield = df.groupby("Dist Code")["Yield"].transform("median")
df["Yield_Lag2"] = df["Yield_Lag2"].fillna(district_median_yield)

df["Yield_Rolling_Mean3"] = (
    df.groupby("Dist Code")["Yield"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

df["Yield_Rolling_Std3"] = (
    df.groupby("Dist Code")["Yield"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).std().fillna(0))
)

# 5. Localized Climate Anomalies (Deviation from district average)
climate_cols = [
    "Avg_Season_Tmax",
    "Total_Season_Rain",
    "Sowing_Rainfall",
    "Terminal_Heat_Tmax",
    "Temperature_Anomaly",
    "Rainfall_Deviation",
]

for col in climate_cols:
    district_mean = df.groupby("Dist Code")[col].transform("mean")
    df[f"{col}_Anomaly"] = df[col] - district_mean

# 6. Climate Interaction Features
eps = 1e-6
df["Terminal_vs_AvgTemp"] = df["Terminal_Heat_Tmax"] - df["Avg_Season_Tmax"]
df["SowingRain_to_TotalRain"] = df["Sowing_Rainfall"] / (df["Total_Season_Rain"] + 1.0)
df["Rain_per_Temp"] = df["Total_Season_Rain"] / (df["Avg_Season_Tmax"] + eps)

# 7. Drop Yield_Lag1 to force the model to learn weather patterns
if "Yield_Lag1" in df.columns:
    df.drop(columns=["Yield_Lag1"], inplace=True)

# 8. Save the new USA moderate dataset
df.to_csv(output_path, index=False)
print(f"Success! USA moderate dataset saved to:\n{output_path}")

Mounted at /content/drive
Success! USA moderate dataset saved to:
/content/drive/MyDrive/wheat_yield_india/dataset/final_usa_wheat_dataset_moderate.csv
